# Анализ ожидаемой продолжительности жизни

Два набора данных, исследующих глобальную продолжительность жизни:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **Ожидаемая продолжительность жизни ВОЗ (WHO)** (2000–2015): 193 страны, 22 показателя (смертность, ИМТ/BMI, ВВП, образование и др.)

Эта книга демонстрирует импорт и анализ файлов CSV на **Python** и **R**.

## 1. Настройка: установка пакетов и загрузка наборов данных

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('Installed pandas + plotly')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Already exists: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Downloaded {name}: {lines} lines")

## 2. Gapminder: Исследование на Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Shape: {gap.shape}")
print(f"Continents: {sorted(gap['continent'].unique())}")
print(f"Year range: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Life expectancy over time by continent
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Life Expectancy by Continent (1952-2007)',
              labels={'lifeExp': 'Life Expectancy (years)', 'year': 'Year'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# GDP vs Life Expectancy (2007), bubble size = population
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='GDP vs Life Expectancy (2007)',
                 labels={'gdpPercap': 'GDP per capita (log)', 'lifeExp': 'Life Expectancy'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: Исследование на R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Life expectancy distribution by continent (boxplot)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Life Expectancy by Continent",
        xlab = "Continent", ylab = "Life Expectancy (years)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# Top 10 countries by life expectancy improvement (1952 vs 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "Top 10: Life Expectancy Gain (1952-2007)",
        xlab = "Years gained",
        col = "#00CC96", border = NA)

## 4. Ожидаемая продолжительность жизни ВОЗ: Исследование на Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Shape: {who.shape}")
print(f"Columns: {list(who.columns)}")
print(f"\nMissing values (top 5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# Developing vs Developed: pre-binned life expectancy distributions
# Explicit bar coordinates render consistently through the browser Plotly bridge.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Life Expectancy: Developing vs Developed',
             labels={'Life expectancy': 'Life Expectancy (years)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Schooling vs Life Expectancy
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Schooling vs Life Expectancy (2014)',
                 labels={'Life expectancy': 'Life Expectancy (years)',
                         'Schooling': 'Years of Schooling'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. Ожидаемая продолжительность жизни ВОЗ: Исследование на R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nCountries:", length(unique(who$Country)))
cat("\nYear range:", range(who$Year))

In [ ]:
# Correlation: Adult Mortality vs Life Expectancy
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Adult Mortality vs Life Expectancy",
     xlab = "Adult Mortality (per 1000)",
     ylab = "Life Expectancy (years)")
legend("topright", legend = c("Developed", "Developing"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Simple linear model: what predicts life expectancy?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Ключевые выводы

- Ожидаемая продолжительность жизни выросла во всем мире, однако между континентами сохраняются значительные различия
- ВВП и уровень образования являются сильными положительными предикторами продолжительности жизни
- Взрослая смертность — наиболее выраженный отрицательный предиктор
- Развивающиеся страны демонстрируют значительно больший разброс показателей